In [1]:
!rm -rf /kaggle/working/*

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_log_error
import warnings
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_log_error
import numpy as np
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.precision', 4)
pd.set_option('display.max_colwidth', None)

In [4]:
n_households = 1500
train_days = 59
test_days = 14

households = [f'H{str(i).zfill(5)}' for i in range(n_households)]

def create_features(n_rows, start_date, days):
    dates = pd.date_range(start=start_date, periods=days).repeat(n_households)
    temp_avg = np.random.normal(15, 5, size=n_rows)
    
    df = pd.DataFrame({
        'household_id': np.tile(households, days),
        'date': dates.strftime('%Y-%m-%d'),
        'num_residents': np.random.randint(1, 7, size=n_rows),
        'home_sqft': np.random.randint(700, 4500, size=n_rows),
        'has_ev': np.random.choice([0, 1], size=n_rows),
        'has_solar': np.random.choice([0, 1], size=n_rows),
        'has_pool': np.random.choice([0, 1], size=n_rows),
        'heating_type': np.random.choice(['electric', 'gas', 'heat_pump'], size=n_rows),
        'hvac_age_years': np.random.randint(1, 26, size=n_rows),
        'temp_avg_c': temp_avg,
        'temp_min_c': temp_avg - np.random.uniform(2, 6, size=n_rows),
        'temp_max_c': temp_avg + np.random.uniform(2, 6, size=n_rows),
        'humidity_pct': np.random.randint(10, 100, size=n_rows),
        'wind_kph': np.random.uniform(0, 30, size=n_rows),
        'precip_mm': np.random.uniform(0, 15, size=n_rows),
        'solar_index': np.random.uniform(0, 10, size=n_rows),
        'is_weekend': (dates.dayofweek >= 5).astype(int),
        'is_holiday': np.random.choice([0, 1], size=n_rows, p=[0.97, 0.03]),
        'prior_day_kwh': np.random.uniform(5, 50, size=n_rows),
        'prior_week_avg_kwh': np.random.uniform(5, 50, size=n_rows)
    })
    return df

train_rows = n_households * train_days
train = create_features(train_rows, '2025-01-01', train_days)
train.insert(0, 'row_id', np.arange(train_rows))
train['kwh'] = np.random.uniform(10, 100, size=train_rows)

test_rows = n_households * test_days
start_row_id = train_rows
test = create_features(test_rows, '2025-03-01', test_days)
test.insert(0, 'row_id', np.arange(start_row_id, start_row_id + test_rows))

sample_sub = pd.DataFrame({
    'row_id': test['row_id'],
    'kwh': np.median(train['kwh'])
})

train.to_csv('train.csv', index=False)
test.to_csv('test.csv', index=False)
sample_sub.to_csv('sample_submission.csv', index=False)

print(f"Train shape: {train.shape} | Test shape: {test.shape} | Sub shape: {sample_sub.shape}")

Train shape: (88500, 22) | Test shape: (21000, 21) | Sub shape: (21000, 2)


In [5]:
train.head()

,row_id,household_id,date,num_residents,home_sqft,has_ev,has_solar,has_pool,heating_type,hvac_age_years,temp_avg_c,temp_min_c,temp_max_c,humidity_pct,wind_kph,precip_mm,solar_index,is_weekend,is_holiday,prior_day_kwh,prior_week_avg_kwh,kwh
0,0,H00000,2025-01-01,4,4347,0,0,0,gas,7,18.1451,16.0969,23.7188,91,28.6533,6.6559,6.3090,0,0,25.4064,41.1728,30.6407
1,1,H00001,2025-01-01,3,3487,0,0,0,heat_pump,16,17.6843,12.3679,20.8964,22,19.7895,11.1211,4.9753,0,0,39.5721,22.7104,75.8149
2,2,H00002,2025-01-01,4,1606,0,1,1,heat_pump,3,18.3198,15.7204,23.7628,70,27.4095,10.8556,4.3478,0,0,12.5201,20.3358,80.4221
3,3,H00003,2025-01-01,6,4480,0,0,1,heat_pump,25,9.2780,5.4063,14.1177,76,1.5253,1.1406,6.8200,0,0,47.0174,42.6926,78.2438
4,4,H00004,2025-01-01,5,3728,1,1,0,heat_pump,3,13.3082,11.0387,18.9177,90,18.4936,1.9897,9.6046,0,0,13.6877,49.2224,86.5626


In [6]:
test.head()

,row_id,household_id,date,num_residents,home_sqft,has_ev,has_solar,has_pool,heating_type,hvac_age_years,temp_avg_c,temp_min_c,temp_max_c,humidity_pct,wind_kph,precip_mm,solar_index,is_weekend,is_holiday,prior_day_kwh,prior_week_avg_kwh
0,88500,H00000,2025-03-01,5,1848,1,1,0,heat_pump,4,10.5072,6.3399,15.2840,52,16.6079,11.1621,2.1787,1,0,42.6084,30.5027
1,88501,H00001,2025-03-01,6,790,0,1,0,gas,21,22.9053,19.3778,28.3283,30,25.7364,4.9748,5.1374,1,0,5.3728,11.4133
2,88502,H00002,2025-03-01,6,817,1,1,0,gas,3,14.9812,9.3203,20.7359,39,7.7932,9.5293,0.6330,1,0,47.6655,32.3933
3,88503,H00003,2025-03-01,1,712,1,1,1,heat_pump,24,8.9851,5.9570,14.4608,22,20.6324,12.9685,1.0212,1,0,9.1980,31.3989
4,88504,H00004,2025-03-01,6,1487,0,0,1,gas,25,12.0353,7.1904,16.0040,58,17.3212,2.7905,1.8173,1,0,20.5573,19.9452


In [7]:
sample_sub.head()

,row_id,kwh
0,88500,55.0348
1,88501,55.0348
2,88502,55.0348
3,88503,55.0348
4,88504,55.0348


In [8]:
sample_sub.shape

(21000, 2)

In [9]:
train.shape

(88500, 22)

In [10]:
train.isnull().sum()

row_id                0
household_id          0
date                  0
num_residents         0
home_sqft             0
has_ev                0
has_solar             0
has_pool              0
heating_type          0
hvac_age_years        0
temp_avg_c            0
temp_min_c            0
temp_max_c            0
humidity_pct          0
wind_kph              0
precip_mm             0
solar_index           0
is_weekend            0
is_holiday            0
prior_day_kwh         0
prior_week_avg_kwh    0
kwh                   0
dtype: int64

In [11]:
test.shape

(21000, 21)

In [12]:
test.isnull().sum()

row_id                0
household_id          0
date                  0
num_residents         0
home_sqft             0
has_ev                0
has_solar             0
has_pool              0
heating_type          0
hvac_age_years        0
temp_avg_c            0
temp_min_c            0
temp_max_c            0
humidity_pct          0
wind_kph              0
precip_mm             0
solar_index           0
is_weekend            0
is_holiday            0
prior_day_kwh         0
prior_week_avg_kwh    0
dtype: int64

In [13]:
train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])

for df in [train, test]:
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    
    df['cdd'] = (df['temp_avg_c'] - 18).clip(lower=0)
    df['hdd'] = (18 - df['temp_avg_c']).clip(lower=0)
    
    df['hvac_cooling_load'] = df['cdd'] * df['home_sqft']
    df['hvac_heating_load'] = df['hdd'] * df['home_sqft']
    
    df['solar_power_potential'] = df['has_solar'] * df['solar_index']
    
    df['temp_range'] = df['temp_max_c'] - df['temp_min_c']

cat_cols = ['heating_type', 'household_id']
for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

features = [c for c in train.columns if c not in ['row_id', 'date', 'kwh']]
target = 'kwh'

X = train[features]
y = train[target]
X_test = test[features]

print(f"Number of features used: {len(features)}")

Number of features used: 28


In [14]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds_lgb = np.zeros(len(X))
test_preds_lgb = np.zeros(len(X_test))

oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

oof_preds_cat = np.zeros(len(X))
test_preds_cat = np.zeros(len(X_test))

y_log = np.log1p(y)

cat_features = list(X.select_dtypes(include=['category', 'object']).columns)

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'max_depth': 6,
    'num_leaves': 31,
    'random_state': 42,
    'verbose': -1,
    'n_estimators': 1500,
    'device_type': 'cpu'
}

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.05,
    'max_depth': 6,
    'random_state': 42,
    'n_estimators': 1500,
    'tree_method': 'hist',
    'device': 'cuda',
    'early_stopping_rounds': 50
}

cat_params = {
    'loss_function': 'RMSE',
    'learning_rate': 0.05,
    'depth': 6,
    'random_seed': 42,
    'verbose': 0,
    'iterations': 1500,
    'task_type': 'GPU',
    'early_stopping_rounds': 50
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
    print(f"--- Training Fold {fold + 1} ---")
    
    X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]
    
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    val_preds_lgb = np.clip(np.expm1(model_lgb.predict(X_val)), 0, None)
    oof_preds_lgb[val_idx] = val_preds_lgb
    test_preds_lgb += np.clip(np.expm1(model_lgb.predict(X_test)), 0, None) / kf.n_splits
    
    model_xgb = xgb.XGBRegressor(**xgb_params, enable_categorical=True)
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    val_preds_xgb = np.clip(np.expm1(model_xgb.predict(X_val)), 0, None)
    oof_preds_xgb[val_idx] = val_preds_xgb
    test_preds_xgb += np.clip(np.expm1(model_xgb.predict(X_test)), 0, None) / kf.n_splits
    
    model_cat = CatBoostRegressor(**cat_params)
    model_cat.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        verbose=False
    )
    val_preds_cat = np.clip(np.expm1(model_cat.predict(X_val)), 0, None)
    oof_preds_cat[val_idx] = val_preds_cat
    test_preds_cat += np.clip(np.expm1(model_cat.predict(X_test)), 0, None) / kf.n_splits

final_oof_preds = (oof_preds_lgb + oof_preds_xgb + oof_preds_cat) / 3
final_test_preds = (test_preds_lgb + test_preds_xgb + test_preds_cat) / 3

cv_score = root_mean_squared_log_error(y, final_oof_preds)
print(f"\nOverall Out-Of-Fold RMSLE: {cv_score:.4f}")

--- Training Fold 1 ---
--- Training Fold 2 ---
--- Training Fold 3 ---
--- Training Fold 4 ---
--- Training Fold 5 ---

Overall Out-Of-Fold RMSLE: 0.5708


In [15]:
import pandas as pd

sub = pd.read_csv('/kaggle/working/sample_submission.csv')

if len(final_test_preds) != len(sub):
    raise ValueError(f"Length mismatch! You have {len(final_test_preds)} predictions, but Kaggle expects {len(sub)}.")

sub['kwh'] = final_test_preds
sub['kwh'] = sub['kwh'].clip(lower=0)

sub = sub.reset_index(drop=True)
sub['row_id'] = sub.index

sub.to_csv('submission_hello.csv', index=False)

print(sub.head())

   row_id      kwh
0       0  47.6487
1       1  47.7525
2       2  47.8112
3       3  47.5480
4       4  48.1339
